In [45]:
import utils_runup
import requests
import pandas as pd
from datetime import datetime

hourly = pd.date_range(start='2025-01-23', end='2025-02-19', freq='D')
formatted_time = hourly.strftime("%Y-%m-%d")


timestamps = formatted_time
twl_region = 1
siteid = 1548
lat, lon = 41.897136303912966, -69.96233057531904
savefolder = "test"

In [53]:
# Get tide from TWL forecast

import os

print('getting tide')

records = []


for timestamp in timestamps:
    d = utils_runup.fetch_water_level_data(timestamp = timestamp,
        region_choice = twl_region, site_id = siteid,
        lat = lat, lon = lon,
        distance_m=500, save_folder= "data_wl")
            
    data_timestamps = pd.to_datetime(d['dateTime'], utc=True)

    TWL_forecast =  d.items()
    target_timestamp = pd.to_datetime(timestamp, utc=True)   
    closest_index = abs(data_timestamps - target_timestamp).argmin()
    
    TWL_forecast = {key: values[closest_index] for key, values in d.items()}
    tide = TWL_forecast['tideWindSetup']
    


    url = "https://vdatum.noaa.gov/vdatumweb/api/convert"


    params = {
        "wlon": lon,
        "wlat": lat,
        "region": "CONTIGUOUS",
        "s_h_frame": "NAD83_2011",
        "s_coor": "utm",
        "s_h_unit": "m",
        "s_v_frame": "LMSL",
        "s_v_unit": "m",
        "s_v_elevation": "height",
        "s_v_geoid": "geoid18",
        "s_z": "0.0",
        "t_h_frame": "NAD83_2011",
        "t_coor": "utm",
        "t_h_unit": "m",
        "t_h_zone": "19",
        "t_v_frame": "NAVD88",
        "t_v_unit": "m",
        "t_v_elevation": "height",
        "t_v_geoid": "geoid18",
        "epoch_in": "0.0",
        "epoch_out": "0.0"
                }


    response = requests.get(url, params=params)
    vdatum_data = response.json()
    print(vdatum_data)
    t_z = float(vdatum_data.get("t_z"))
    tide = tide + t_z
    TWL_forecast['twl'] += t_z
    TWL_forecast['twl05'] += t_z
    TWL_forecast['twl95'] += t_z
    TWL_forecast['tideWindSetup'] += t_z

    records.append(TWL_forecast)

df = pd.DataFrame(records)
print(df.head())

getting tide
Boston, MA
{'errorCode': 412, 'message': 'Source Horizontal Zone is empty!'}


TypeError: float() argument must be a string or a real number, not 'NoneType'

In [ ]:
d = utils_runup.fetch_water_level_data(timestamp = "2025-01-23",
        region_choice = 1, site_id = 1548,
        lat = lat, lon = lon,
        distance_m=500, save_folder= "data_wl")

In [ ]:
import matplotlib.pyplot as plt

print(d)
plt.plot(d['dateTime'], d['twl'], label='TWL')
plt.xlabel('Date')
plt.ylabel('TWL')
plt.title('Tide Water Level Forecast')
plt.legend()
plt.show()
